In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nagalaxmip/keypart5/HumanEval_KEY.xlsx
/kaggle/input/datasets/nagalaxmip/external-test-set/FLORES200_en_te_devtest_UTF8.csv
/kaggle/input/datasets/nagalaxmip/external-test-set/IN22_Conv_EN_TE.xlsx
/kaggle/input/datasets/nagalaxmip/external-test-set/IN22_Gen_en_te.csv
/kaggle/input/datasets/nagalaxmip/bpccpart5pscab/bpcc_telugu_fixed100k samples.csv
/kaggle/input/datasets/nagalaxmip/excel-2-reviewers-rating/HumanEval_Rater1_LikhithaReddy_ExpertReview.xlsx
/kaggle/input/datasets/nagalaxmip/excel-2-reviewers-rating/HumanEval_Rater1guideCompleted-.xlsx
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/__notebook__.ipynb
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/__output__.json
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/__results__.html
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/custom.css
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/FINAL_EVIDENCE_SUMMARY.txt
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/PAPER_ASSETS.md
/kag

In [6]:
import glob, os, shutil
os.makedirs("human_eval", exist_ok=True)
for f in glob.glob("/kaggle/input/**/HumanEval_*.xlsx", recursive=True):
    shutil.copy(f, "human_eval/")
print(sorted(os.listdir("human_eval")))

['HumanEval_KEY.xlsx', 'HumanEval_Rater1.xlsx', 'HumanEval_Rater1_LikhithaReddy_ExpertReview.xlsx', 'HumanEval_Rater1guideCompleted-.xlsx', 'HumanEval_Rater2.xlsx', 'HumanEval_Rater3.xlsx', 'results']


In [7]:
"""
================================================================================
PART 5B - ANALYSE THE RETURNED HUMAN-EVALUATION WORKBOOKS  (one cell)
================================================================================
Put the filled files HumanEval_Rater1.xlsx ... (any file names containing 'Rater') and
HumanEval_KEY.xlsx in the folder human_eval/ and run this cell. It writes human_eval/results/:
  HE1_quality_control.csv      QC pass rate and repeat consistency per rater
  HE2_agreement.csv            Krippendorff alpha (ordinal) for adequacy/fluency/marker accuracy,
                               Fleiss kappa for acceptability
  HE3_system_scores.csv        mean adequacy, fluency, acceptability, marker accuracy per system
                               (main items only, 95% bootstrap CI over items)
  HE4_paired_differences.csv   PS-CAB - baseline, CAPM - baseline (bootstrap CI, Wilcoxon p, Holm)
  HE5_mixed_model.txt          adequacy ~ system + (1 | item), rater as variance component
  HE6_error_types.csv          main marker error distribution per system
  HE7_by_stratum.csv           scores by stratum (BPCC with/without prepositions, IN22-Conv)
Only 'main' items enter HE3-HE7; repeats and QC items are used for HE1 only.
================================================================================
"""
OUT = "human_eval"
SYSTEMS = ["baseline_s42", "pscab_s42", "capm_s42"]
BASE = "baseline_s42"
N_BOOT, SEED = 2000, 12345

import os, glob, subprocess, sys
import numpy as np, pandas as pd
for pkg in ("krippendorff", "statsmodels", "openpyxl"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import krippendorff
from scipy.stats import wilcoxon
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

RES = f"{OUT}/results"; os.makedirs(RES, exist_ok=True)
key = pd.read_excel(f"{OUT}/HumanEval_KEY.xlsx", sheet_name="key")
files = sorted(f for f in glob.glob(f"{OUT}/*Rater*.xlsx"))
if not files:
    raise FileNotFoundError("No rater workbooks found in human_eval/")
rows = []
for k, f in enumerate(files, 1):
    xl = pd.ExcelFile(f)
    sh = [s for s in xl.sheet_names if s.lower().startswith("rater")][0]
    d = xl.parse(sh)
    d = d.rename(columns={"Adequacy (1-5)": "adequacy", "Fluency (1-5)": "fluency",
                          "Markers correct (number)": "m_correct", "Main marker error": "m_error",
                          "Acceptable (Y/N)": "acceptable", "Prepositions highlighted (n)": "n_prep_shown"})
    d["rater"] = f"R{k}"
    rows.append(d[["Item", "Slot", "rater", "adequacy", "fluency", "m_correct", "m_error", "acceptable",
                   "n_prep_shown"]])
H = pd.concat(rows).merge(key, on=["Item", "Slot"], how="left")
for c in ("adequacy", "fluency", "m_correct"):
    H[c] = pd.to_numeric(H[c], errors="coerce")          # tolerate "4 " typed as text
H["acceptable"] = H["acceptable"].astype(str).str.strip().str.upper().replace({"YES": "Y", "NO": "N"})

# ---- stop early if the workbooks have not been rated yet
fill = H.groupby("rater")["adequacy"].apply(lambda x: x.notna().mean())
print("[check] share of rows rated per rater:", fill.round(3).to_dict())
def _no_ratings():
    print("\n*** NO RATINGS YET - nothing to analyse. ***\n"
          "The files in human_eval/ are the BLANK workbooks made by Part 5A.\n"
          "1) Download HumanEval_Rater1/2/3.xlsx and give one to each rater.\n"
          "2) Raters fill the yellow columns (Adequacy, Fluency, Markers correct, Main marker error, Acceptable).\n"
          "3) Upload the FILLED files + HumanEval_KEY.xlsx as a Kaggle dataset, copy them into human_eval/, run again.")

def _run_analysis(H):
    if (fill < 0.95).any():
        print("[warning] some raters have not finished; incomplete rows are dropped from each analysis.")
    H["acc_bin"] = H["acceptable"].map({"Y": 1, "N": 0})
    H["marker_acc"] = np.where(H["n_prep"] > 0, H["m_correct"] / H["n_prep"].replace(0, np.nan), np.nan)
    print(f"[load] {len(files)} raters, {len(H)} judgements; missing adequacy: {H.adequacy.isna().sum()}")

    # ---------------------------------------------------------------- HE1 quality control
    qc = []
    for r, d in H.groupby("rater"):
        q = d[d.qc_broken == 1]
        rep = d[d.kind.isin(["main", "repeat"])]
        pair = rep.pivot_table(index=["split", "row", "system"], columns="kind", values="adequacy", aggfunc="first").dropna()
        qc.append({"rater": r, "qc_items": len(q), "qc_pass_rate(adequacy<=2)": round((q.adequacy <= 2).mean(), 3),
                   "repeat_pairs": len(pair),
                   "repeat_exact_agreement": round((pair["main"] == pair["repeat"]).mean(), 3) if len(pair) else np.nan,
                   "repeat_within_1": round(((pair["main"] - pair["repeat"]).abs() <= 1).mean(), 3) if len(pair) else np.nan})
    pd.DataFrame(qc).to_csv(f"{RES}/HE1_quality_control.csv", index=False)

    M = H[H.kind == "main"].copy()
    M["unit"] = M["Item"] + "_" + M["system"]

    # ---------------------------------------------------------------- HE2 agreement
    ag = []
    for col in ("adequacy", "fluency", "marker_acc"):
        mat = M.pivot_table(index="rater", columns="unit", values=col, aggfunc="first")
        lvl = "interval" if col == "marker_acc" else "ordinal"
        try:
            a = krippendorff.alpha(reliability_data=mat.values.astype(float), level_of_measurement=lvl)
        except ValueError as e:                                # e.g. all raters gave one value only
            print(f"[HE2] alpha for {col} not computable: {e}"); a = np.nan
        ag.append({"measure": col, "statistic": f"Krippendorff alpha ({lvl})", "value": round(a, 3), "units": mat.shape[1]})
    acc = M.pivot_table(index="unit", columns="rater", values="acc_bin", aggfunc="first").dropna()
    if len(acc):
        tab, _ = aggregate_raters(acc.values.astype(int))
        ag.append({"measure": "acceptable", "statistic": "Fleiss kappa", "value": round(fleiss_kappa(tab), 3),
                   "units": len(acc)})
    pd.DataFrame(ag).to_csv(f"{RES}/HE2_agreement.csv", index=False)

    # per item x system: average over raters
    U = M.groupby(["Item", "split", "row", "stratum", "system"]).agg(
        adequacy=("adequacy", "mean"), fluency=("fluency", "mean"), acceptable=("acc_bin", "mean"),
        m_correct=("m_correct", "mean"), n_prep=("n_prep", "first")).reset_index()
    rng = np.random.default_rng(SEED)
    items = U.Item.unique()
    W = {m: U.pivot(index="Item", columns="system", values=m) for m in ("adequacy", "fluency", "acceptable", "m_correct")}
    NP = U.drop_duplicates("Item").set_index("Item")["n_prep"]


    def stat(metric, sysname, idx):
        if metric == "marker_accuracy":
            nn = NP.loc[idx]
            keep = nn > 0
            return W["m_correct"].loc[idx][sysname][keep.values].sum() / nn[keep].sum()
        return W[metric].loc[idx][sysname].mean()


    sc = []
    B = [rng.choice(items, len(items), replace=True) for _ in range(N_BOOT)]
    for s in SYSTEMS:
        for m in ("adequacy", "fluency", "acceptable", "marker_accuracy"):
            pt = stat(m, s, items)
            bs = [stat(m, s, b) for b in B]
            sc.append({"system": s, "metric": m, "mean": round(pt, 4),
                       "ci95_low": round(np.percentile(bs, 2.5), 4), "ci95_high": round(np.percentile(bs, 97.5), 4),
                       "items": len(items)})
    pd.DataFrame(sc).to_csv(f"{RES}/HE3_system_scores.csv", index=False)

    # ---------------------------------------------------------------- HE4 paired differences
    pdif = []
    for s in SYSTEMS:
        if s == BASE:
            continue
        for m in ("adequacy", "fluency", "acceptable", "marker_accuracy"):
            pt = stat(m, s, items) - stat(m, BASE, items)
            bs = [stat(m, s, b) - stat(m, BASE, b) for b in B]
            if m == "marker_accuracy":
                keep = NP[NP > 0].index
                x = (W["m_correct"].loc[keep, s] - W["m_correct"].loc[keep, BASE]) / NP[keep]
            else:
                x = W[m][s] - W[m][BASE]
            x = x.dropna()
            p = wilcoxon(x).pvalue if (x != 0).any() else 1.0
            pdif.append({"comparison": f"{s} - {BASE}", "metric": m, "delta": round(pt, 4),
                         "ci95_low": round(np.percentile(bs, 2.5), 4), "ci95_high": round(np.percentile(bs, 97.5), 4),
                         "wilcoxon_p": round(p, 4)})
    D = pd.DataFrame(pdif)
    D["p_holm"] = multipletests(D["wilcoxon_p"], method="holm")[1].round(4)
    D.to_csv(f"{RES}/HE4_paired_differences.csv", index=False)

    # ---------------------------------------------------------------- HE5 mixed model
    M["system_c"] = pd.Categorical(M["system"], categories=SYSTEMS)
    try:
        md = smf.mixedlm("adequacy ~ C(system_c)", M.dropna(subset=["adequacy"]), groups="Item",
                         vc_formula={"rater": "0 + C(rater)"}).fit(reml=True)
        txt = md.summary().as_text()
    except Exception as e:
        txt = f"mixed model failed: {e}"
    open(f"{RES}/HE5_mixed_model.txt", "w").write(txt)

    # ---------------------------------------------------------------- HE6 error types, HE7 strata
    M["m_error"] = M["m_error"].fillna("(blank)")
    M.groupby(["system", "m_error"]).size().unstack(fill_value=0).to_csv(f"{RES}/HE6_error_types.csv")
    U["marker_acc_item"] = U["m_correct"] / U["n_prep"].replace(0, np.nan)
    U.groupby(["stratum", "system"]).agg(adequacy=("adequacy", "mean"), fluency=("fluency", "mean"),
                                         acceptable=("acceptable", "mean"),
                                         marker_acc=("marker_acc_item", "mean"), items=("Item", "nunique")
                                         ).round(4).to_csv(f"{RES}/HE7_by_stratum.csv")
    for f in sorted(glob.glob(f"{RES}/HE*.csv")):
        print("\n==", os.path.basename(f)); print(pd.read_csv(f).to_string(index=False))
    print("\n", txt[:1500])


if fill.max() == 0:
    _no_ratings()
else:
    _run_analysis(H)

[check] share of rows rated per rater: {'R1': 0.0, 'R2': 1.0, 'R3': 1.0, 'R4': 0.0, 'R5': 0.0}
[warning] some raters have not finished; incomplete rows are dropped from each analysis.
[load] 5 raters, 5175 judgements; missing adequacy: 3105

== HE1_quality_control.csv
rater  qc_items  qc_pass_rate(adequacy<=2)  repeat_pairs  repeat_exact_agreement  repeat_within_1
   R1        15                        0.0             0                     NaN              NaN
   R2        15                        1.0            90                   1.000              1.0
   R3        15                        1.0            90                   0.956              1.0
   R4        15                        0.0             0                     NaN              NaN
   R5        15                        0.0             0                     NaN              NaN

== HE2_agreement.csv
   measure                     statistic  value  units
  adequacy  Krippendorff alpha (ordinal)  0.680    900
   fluency 

In [8]:
"""
================================================================================
PART 5B-2 -- HUMAN EVALUATION, PER-JUDGE ANALYSIS (Supplementary S11)
================================================================================
Run this cell directly after the Part 5B analysis cell in the same session (it reuses
OUT, SYSTEMS, BASE, N_BOOT, SEED and the merged judgement table H built there).
Only the 300 main items are used. For EACH judge separately it writes to
human_eval/results/per_judge/:
  HJ3_system_scores_<judge>.csv     mean + 95% item-level bootstrap CI (2,000 resamples)
  HJ4_paired_differences_<judge>.csv  system - baseline, bootstrap CI, Wilcoxon p, Holm over 8
  HJ5_mixed_model_<judge>.txt       adequacy ~ system + (1 | item)   (REML)
  HJ7_by_stratum_<judge>.csv        stratum means and system - baseline differences
and HJ7_by_stratum_pooled.csv (mean over both judges, same as Part 5B HE7 + differences).
Judge names follow the workbook file names: R1 = first file in sorted order.
================================================================================
"""
import os, glob
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

assert "H" in globals() and "SYSTEMS" in globals(), "Run the Part 5B analysis cell first in this session."
PJ = f"{OUT}/results/per_judge"
os.makedirs(PJ, exist_ok=True)
files = sorted(glob.glob(f"{OUT}/*Rater*.xlsx"))
NAMES = {f"R{k}": os.path.basename(f).replace(".xlsx", "") for k, f in enumerate(files, 1)}
print("[judges]", NAMES)

H["acc_bin"] = H["acceptable"].map({"Y": 1, "N": 0})
MAIN = H[H.kind == "main"].copy()
MEASURES = ("adequacy", "fluency", "acceptable", "marker_accuracy")


def tables(M):
    U = M.groupby(["Item", "stratum", "system"]).agg(
        adequacy=("adequacy", "mean"), fluency=("fluency", "mean"), acceptable=("acc_bin", "mean"),
        m_correct=("m_correct", "mean"), n_prep=("n_prep", "first")).reset_index()
    W = {m: U.pivot(index="Item", columns="system", values=m) for m in ("adequacy", "fluency", "acceptable", "m_correct")}
    NP = U.drop_duplicates("Item").set_index("Item")["n_prep"]
    return U, W, NP


def stat(W, NP, metric, s, idx):
    if metric == "marker_accuracy":
        nn = NP.loc[idx]
        keep = (nn > 0).values
        return W["m_correct"].loc[idx][s].values[keep].sum() / nn.values[keep].sum()
    return W[metric].loc[idx][s].mean()


def strata(U, label):
    U = U.copy()
    U["marker_acc_item"] = U["m_correct"] / U["n_prep"].replace(0, np.nan)
    S = U.groupby(["stratum", "system"]).agg(adequacy=("adequacy", "mean"), fluency=("fluency", "mean"),
                                             acceptable=("acceptable", "mean"), marker_acc=("marker_acc_item", "mean"),
                                             items=("Item", "nunique")).round(4).reset_index()
    rows = []
    for st, g in S.groupby("stratum"):
        b = g[g.system == BASE].iloc[0]
        for s in SYSTEMS:
            if s == BASE:
                continue
            r = g[g.system == s].iloc[0]
            rows.append({"judge": label, "stratum": st, "comparison": f"{s} - {BASE}", "items": int(r["items"]),
                         **{f"d_{m}": round(r[m] - b[m], 4) for m in ("adequacy", "fluency", "acceptable", "marker_acc")}})
    return S, pd.DataFrame(rows)


summary = []
for judge, M in MAIN.groupby("rater"):
    name = NAMES.get(judge, judge)
    U, W, NP = tables(M)
    items = U.Item.unique()
    rng = np.random.default_rng(SEED)
    B = [rng.choice(items, len(items), replace=True) for _ in range(N_BOOT)]

    sc = []
    for s in SYSTEMS:
        for m in MEASURES:
            bs = [stat(W, NP, m, s, b) for b in B]
            sc.append({"judge": name, "system": s, "metric": m, "mean": round(stat(W, NP, m, s, items), 4),
                       "ci95_low": round(np.percentile(bs, 2.5), 4), "ci95_high": round(np.percentile(bs, 97.5), 4)})
    pd.DataFrame(sc).to_csv(f"{PJ}/HJ3_system_scores_{judge}.csv", index=False)

    pdif = []
    for s in SYSTEMS:
        if s == BASE:
            continue
        for m in MEASURES:
            bs = [stat(W, NP, m, s, b) - stat(W, NP, m, BASE, b) for b in B]
            if m == "marker_accuracy":
                keep = NP[NP > 0].index
                x = (W["m_correct"].loc[keep, s] - W["m_correct"].loc[keep, BASE]) / NP[keep]
            else:
                x = W[m][s] - W[m][BASE]
            x = x.dropna()
            p = wilcoxon(x).pvalue if (x != 0).any() else 1.0
            pdif.append({"judge": name, "comparison": f"{s} - {BASE}", "metric": m,
                         "delta": round(stat(W, NP, m, s, items) - stat(W, NP, m, BASE, items), 4),
                         "ci95_low": round(np.percentile(bs, 2.5), 4), "ci95_high": round(np.percentile(bs, 97.5), 4),
                         "wilcoxon_p": round(p, 4)})
    D = pd.DataFrame(pdif)
    D["p_holm"] = multipletests(D["wilcoxon_p"], method="holm")[1].round(4)
    D.to_csv(f"{PJ}/HJ4_paired_differences_{judge}.csv", index=False)

    M = M.dropna(subset=["adequacy"]).copy()
    M["system_c"] = pd.Categorical(M["system"], categories=SYSTEMS)
    try:
        fit = smf.mixedlm("adequacy ~ C(system_c)", M, groups="Item").fit(reml=True)
        txt = fit.summary().as_text()
        ci = fit.conf_int()
        for s in SYSTEMS[1:]:
            k = f"C(system_c)[T.{s}]"
            summary.append({"judge": name, "model": "per-judge (item random intercept)", "term": f"{s} - {BASE}",
                            "coef": round(fit.params[k], 4), "ci95_low": round(ci.loc[k, 0], 4),
                            "ci95_high": round(ci.loc[k, 1], 4), "p": round(fit.pvalues[k], 4)})
    except Exception as e:
        txt = f"mixed model failed: {e}"
    open(f"{PJ}/HJ5_mixed_model_{judge}.txt", "w").write(f"judge: {name}\n\n{txt}")

    S, Sd = strata(U, name)
    S.to_csv(f"{PJ}/HJ7_by_stratum_{judge}.csv", index=False)
    Sd.to_csv(f"{PJ}/HJ7_stratum_differences_{judge}.csv", index=False)
    print(f"\n######## {name} ({judge})")
    print(pd.DataFrame(sc).to_string(index=False))
    print(D.to_string(index=False))
    print(Sd.to_string(index=False))

# pooled mixed model coefficients (same model as Part 5B HE5) for the summary table
M = MAIN.dropna(subset=["adequacy"]).copy()
M["system_c"] = pd.Categorical(M["system"], categories=SYSTEMS)
fit = smf.mixedlm("adequacy ~ C(system_c)", M, groups="Item", vc_formula={"rater": "0 + C(rater)"}).fit(reml=True)
ci = fit.conf_int()
for s in SYSTEMS[1:]:
    k = f"C(system_c)[T.{s}]"
    summary.append({"judge": "both (pooled)", "model": "item random intercept + judge variance component",
                    "term": f"{s} - {BASE}", "coef": round(fit.params[k], 4), "ci95_low": round(ci.loc[k, 0], 4),
                    "ci95_high": round(ci.loc[k, 1], 4), "p": round(fit.pvalues[k], 4)})
pd.DataFrame(summary).to_csv(f"{PJ}/HJ5_mixed_models_summary.csv", index=False)

U, _, _ = tables(MAIN)
S, Sd = strata(U, "both (mean of judges)")
S.to_csv(f"{PJ}/HJ7_by_stratum_pooled.csv", index=False)
Sd.to_csv(f"{PJ}/HJ7_stratum_differences_pooled.csv", index=False)
print("\n######## mixed models\n" + pd.DataFrame(summary).to_string(index=False))
print("\n######## strata, mean of both judges\n" + Sd.to_string(index=False))
print(f"\n[saved] {PJ}/  -> add this folder to Supplementary S11")

[judges] {'R1': 'HumanEval_Rater1', 'R2': 'HumanEval_Rater1_LikhithaReddy_ExpertReview', 'R3': 'HumanEval_Rater1guideCompleted-', 'R4': 'HumanEval_Rater2', 'R5': 'HumanEval_Rater3'}

######## HumanEval_Rater1 (R1)
           judge       system          metric  mean  ci95_low  ci95_high
HumanEval_Rater1 baseline_s42        adequacy   NaN       NaN        NaN
HumanEval_Rater1 baseline_s42         fluency   NaN       NaN        NaN
HumanEval_Rater1 baseline_s42      acceptable   NaN       NaN        NaN
HumanEval_Rater1 baseline_s42 marker_accuracy   NaN       NaN        NaN
HumanEval_Rater1    pscab_s42        adequacy   NaN       NaN        NaN
HumanEval_Rater1    pscab_s42         fluency   NaN       NaN        NaN
HumanEval_Rater1    pscab_s42      acceptable   NaN       NaN        NaN
HumanEval_Rater1    pscab_s42 marker_accuracy   NaN       NaN        NaN
HumanEval_Rater1     capm_s42        adequacy   NaN       NaN        NaN
HumanEval_Rater1     capm_s42         fluency   NaN     